In [ ]:
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 19.8 MB/s eta 0:00:00


In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = urllib.request.urlopen(url)
shakespear_data = response.read().decode('utf-8')

with open('data.txt','w') as file:
  file.write(shakespear_data)


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordPieceTrainer

tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

trainer = WordPieceTrainer(
    vocab_size=8000,
    special_tokens=["[UNK]", "[PAD]"]
)

tokenizer.train(["data.txt"], trainer)

In [ ]:
import torch
from torch.utils.data import Dataset

class ShakespeareDataset(Dataset):
    def __init__(self, text, tokenizer, seq_len=128, stride=64):
        self.seq_len = seq_len
        self.stride = stride
        self.samples = []

        encoding = tokenizer.encode(text)
        token_ids = encoding.ids

        for i in range(0, len(token_ids) - seq_len, stride):
            chunk = token_ids[i:i + seq_len + 1]

            if len(chunk) == seq_len + 1:
                self.samples.append(chunk)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        chunk = self.samples[idx]

        input_ids = torch.tensor(chunk[:-1], dtype=torch.long)
        target_ids = torch.tensor(chunk[1:], dtype=torch.long)

        return input_ids, target_ids

In [ ]:
import math
import torch
import torch.nn as nn
import pytorch_lightning as pl


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, Section 3.5 of the paper."""
    def __init__(self, embedding_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embedding_dim) #[5000,512]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) #[1,5000]
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() *
                              (-math.log(10000.0) / embedding_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, embedding_dim]

    def forward(self, x):
        # x: [batch, seq_len, embedding_dim]
        return x + self.pe[:, :x.size(1), :]


class Transformer(pl.LightningModule):
    def __init__(self, vocab_size, embedding_dim, num_heads, lr):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.pos_encoding = PositionalEncoding(embedding_dim)
        self.dropout = nn.Dropout(0.1)

        self.masked_attn = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=num_heads, batch_first=True)
        self.masked_norm = nn.LayerNorm(normalized_shape=embedding_dim)

        self.attn = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(normalized_shape=embedding_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(in_features=embedding_dim, out_features=1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(in_features=1024, out_features=embedding_dim),
        )
        self.ff_norm = nn.LayerNorm(normalized_shape=embedding_dim)

        self.lr = lr
        self.linear_layer = nn.Linear(in_features=embedding_dim, out_features=vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, x):
        embedding = self.embedding(x)
        embedding = self.pos_encoding(embedding)
        embedding = self.dropout(embedding)

        seq_len = x.size(1)
        # causal mask: True = "not allowed to attend"
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1
        )

        masked_out, _ = self.masked_attn(embedding, embedding, embedding, attn_mask=causal_mask)
        x1 = self.masked_norm(embedding + masked_out)          # residual + norm

        attn_out, _ = self.attn(x1, x1, x1, attn_mask=causal_mask)
        x2 = self.norm(x1 + attn_out)                           # residual + norm

        ff_out = self.feed_forward(x2)
        x3 = self.ff_norm(x2 + ff_out)                          # residual + norm

        logits = self.linear_layer(x3)                          # raw logits, no softmax here
        return logits

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.lr, betas=(0.9, 0.98), eps=1e-9
        )
        return optimizer


In [ ]:
from torch.utils.data import DataLoader, random_split
from pytorch_lightning import Trainer

dataset = ShakespeareDataset(
    shakespear_data,
    tokenizer,
    seq_len=128,
    stride=64
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128,shuffle=False)

model = Transformer(vocab_size = 8000,embedding_dim=512,num_heads=8,lr=1e-9)

trainer = pl.Trainer(
    max_epochs=30,
    devices = "auto"
)

trainer.fit(model,train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name         ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ embedding    │ Embedding          │  4.1 M │ train │     0 │
│ 1  │ pos_encoding │ PositionalEncoding │      0 │ train │     0 │
│ 2  │ dropout      │ Dropout            │      0 │ train │     0 │
│ 3  │ masked_attn  │ MultiheadAttention │  1.1 M │ train │     0 │
│ 4  │ masked_norm  │ LayerNorm          │  1.0 K │ train │     0 │
│ 5  │ attn         │ MultiheadAttention │  1.1 M │ train │     0 │
│ 6  │ norm         │ LayerNorm          │  1.0 K │ train │     0 │
│ 7  │ feed_forward │ Sequential         │  1.1 M │ train │     0 │
│ 8  │ ff_norm      │ LayerNorm          │  1.0 K │ train │     0 │
│ 9  │ linear_layer │ Linear             │  4.1 M │ train │     0 │
│ 10 │ loss_fn      │ CrossEntropyLoss   │      0 │ train │     0 │
└────┴──────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 11.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.4 M                                                                                               
Total estimated model params size (MB): 45.418                                                                     
Modules in train mode: 17                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches 
(28) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if
you want to see logs for the training epoch.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


In [ ]:
import torch

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, device="cuda"):
    model.eval()
    model.to(device)

    encoding = tokenizer.encode(prompt)
    input_ids = torch.tensor([encoding.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        logits = model(input_ids)

        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    generated_ids = input_ids.squeeze(0).tolist()

    return tokenizer.decode(generated_ids)

# usage
output = generate(model, tokenizer, "Once upon a time", max_new_tokens=50, device="cuda")
print(output)

Once upon a time Once ##illing ##ruly weather skill concerns ##ires concerns ##ires concerns ##ires concerns ##ires concerns ##rum nobles imperial fox goss minute du ver ##F noble ##unts ##enty ##UT ##roop frozen ##iece Biondello VOL quench nose bridal ##BOW unruly making Never amazement leave graft sings Last spr concerns ##olve Unless rites conscience
